# Detección de Personas y Objetos con Faster R-CNN
Este notebook muestra cómo usar un modelo preentrenado de torchvision para detectar objetos en imágenes sin necesidad de entrenamiento.

In [ ]:
# Instalación para Google Colab (descomentar si es necesario)
# !pip install torch torchvision matplotlib
# !apt-get install -y libglib2.0-0 libsm6 libxrender-dev libxext6

## 1. Cargar el modelo preentrenado Faster R-CNN

In [ ]:
import torch
import torchvision
from torchvision.transforms import functional as F
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
model.eval().to(device)

## 2. Cargar una imagen de prueba

In [ ]:
# Subir imagen
image_path = "example.jpg"  # Reemplazar por una imagen válida
image = Image.open(image_path).convert("RGB")
image_tensor = F.to_tensor(image).to(device)

## 3. Inferencia y visualización

In [ ]:
with torch.no_grad():
    prediction = model([image_tensor])[0]

COCO_INSTANCE_CATEGORY_NAMES = [ '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train',
    'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog',
    'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase',
    'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard',
    'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed',
    'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven',
    'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']

fig, ax = plt.subplots(1, figsize=(12, 9))
ax.imshow(image)

for box, label, score in zip(prediction['boxes'], prediction['labels'], prediction['scores']):
    if score > 0.6:
        box = box.to("cpu")
        xmin, ymin, xmax, ymax = box
        rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                 linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(xmin, ymin, f"{COCO_INSTANCE_CATEGORY_NAMES[label]}: {score:.2f}",
                bbox=dict(facecolor='yellow', alpha=0.5))

plt.axis('off')
plt.show()